In [ ]:
import math
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def quantum_random(min_value, max_value):
    """
    Generate a uniformly random integer in [min_value, max_value].
    Uses rejection sampling when the range isn't a power of 2.
    """
    n_outcomes = max_value - min_value + 1

    # Find smallest power of 2 >= n_outcomes
    n_qubits = math.ceil(math.log2(n_outcomes))
    max_state = 2 ** n_qubits  # e.g., range 5 → 3 qubits → max_state = 8

    simulator = AerSimulator()

    while True:
        # Build and run circuit
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        qc.measure(range(n_qubits), range(n_qubits))

        result = simulator.run(qc, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]
        value = int(bitstring, 2)

        # Rejection: if value falls outside our desired range, retry
        if value < n_outcomes:
            return min_value + value
        # else: discard and try again (expected retries < 2×)


# --- Test ---
from collections import Counter

# Test 1: power-of-2 range (no rejection needed)
results = [quantum_random(0, 7) for _ in range(1000)]
print("Range [0,7]:", dict(sorted(Counter(results).items())))

# Test 2: non-power-of-2 range (rejection kicks in)
results = [quantum_random(0, 4) for _ in range(1000)]
print("Range [0,4]:", dict(sorted(Counter(results).items())))

# Test 3: arbitrary range
results = [quantum_random(10, 14) for _ in range(1000)]
print("Range [10,14]:", dict(sorted(Counter(results).items())))

Range [0,7]: {0: 134, 1: 132, 2: 124, 3: 111, 4: 123, 5: 137, 6: 104, 7: 135}
Range [0,4]: {0: 203, 1: 183, 2: 191, 3: 207, 4: 216}
Range [10,14]: {10: 220, 11: 189, 12: 194, 13: 196, 14: 201}


This code implements a quantum random number generator (QRNG) that produces uniform random integers within any given range [min_value, max_value].
It achieves true randomness by putting quantum bits (qubits) into a state of superposition and measuring them. Because the number of possible states from a set of qubits is always a power of 2, it uses a technique called rejection sampling to cleanly map those states onto arbitrary ranges.
Here is a step-by-step breakdown of how the solution works:

1. Determining the Required Qubits

To cover a range of numbers, the code first calculates how many unique outcomes are needed: n_outcomes = max_value - min_value + 1.
• If you want a number between 10 and 14, there are 5 possible outcomes (10, 11, 12, 13, 14).
• It calculates the smallest power of 2 that can accommodate this range using math.ceil(math.log2(n_outcomes)). For 5 outcomes, it calculates \(\lceil\log_2(5)\rceil = \lceil2.32\rceil = 3\) qubits.
• 3 qubits can represent 2³ = 8 unique binary states (000 to 111).

2. Generating the Quantum Random Bitstring

Inside the while True: loop, a new quantum circuit is constructed and executed:
• qc.h(range(n_qubits)): This applies a Hadamard (H) gate to every qubit. The Hadamard gate places a qubit initially at 0 into a perfect 50/50 superposition of 0 and 1.
• qc.measure(...): Measuring the qubits forces them to collapse out of superposition into concrete classical bits (0 or 1). Because of the superposition, every possible bitstring combination has an exactly equal probability of occurring.
• simulator.run(qc, shots=1): The code samples the circuit exactly once using AerSimulator to get a single random bitstring (e.g., '101'), which is then converted into a base-10 integer (int('101', 2) -> 5).

3. Rejection Sampling (Handling Non-Powers of 2)

Because 3 qubits generate numbers from 0 to 7 uniformly, but we only want 5 outcomes (indexed as 0 to 4 relative to our minimum), the code must handle the "extra" numbers (5, 6, and 7).
• if value < n_outcomes:: If the generated integer falls within the valid index range (0 to 4), it is accepted. The final number is shifted by the minimum bounds and returned: return min_value + value.
• else:: If the integer is 5, 6, or 7, it falls outside our range. The code silently discards (rejects) the result, the loop restarts, and it samples the quantum circuit again.

Why this approach is robust

• Perfect Uniformity: Rejection sampling guarantees that every valid number in your range has an exactly equal chance of being picked. Simply using the modulo operator (e.g., value % n_outcomes) on a power-of-2 state would introduce modulo bias, making smaller numbers slightly more likely to appear than larger ones.
• Efficiency: Because it always picks the smallest power of 2 greater than or equal to the range, the algorithm has a worst-case efficiency where it will discard fewer than half of the samples. On average, it requires fewer than 2 attempts to find a valid number.
